In [ ]:
%%capture
!git clone --single-branch --branch fast_tokenizers_BARTpho_PhoBERT_BERTweet https://github.com/datquocnguyen/transformers.git

In [ ]:
cd transformers

In [ ]:
%%capture
!pip3 install -e .

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

In [ ]:
tokenizer.is_fast

In [ ]:
import os
import ast
from datasets import load_dataset, Dataset
import pandas as pd
from huggingface_hub import login

login("your_huggingface_auth_token_here")

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
df_train = pd.read_csv("./datasets/ViLegalNLI/ViLegalNLI_train.csv")
df_val = pd.read_csv("./datasets/ViLegalNLI/ViLegalNLI_val.csv")
df_test = pd.DataFrame(load_dataset("VLSP2025-LegalSML/Public-Test", "nli_questions")["train"])
len(df_train), len(df_val), len(df_test)

In [ ]:
df_test = df_test[["legal_document", "specific_question", "answer"]]
df_test.columns = ["context", "question", "label"]
df_test["label"] = df_test["label"].apply(lambda x: "Có" if x == 0 else "Không")
df_test

In [ ]:
df_train["label"].value_counts()

In [ ]:
df_val["label"].value_counts()

In [ ]:
df_test["label"].value_counts()

In [ ]:
df_train["option_A"] = "Có"
df_train["option_B"] = "Không"

df_val["option_A"] = "Có"
df_val["option_B"] = "Không"

df_test["option_A"] = "Có"
df_test["option_B"] = "Không"

In [ ]:
df_train = df_train[["context", "question", "option_A", "option_B", "label"]]
df_val = df_val[["context", "question", "option_A", "option_B", "label"]]
df_test = df_test[["context", "question", "option_A", "option_B", "label"]]

In [ ]:
df_test

In [ ]:
df_train.columns = ["sent1", "sent2", "ending0", "ending1", "label"]
df_val.columns = ["sent1", "sent2", "ending0", "ending1", "label"]
df_test.columns = ["sent1", "sent2", "ending0", "ending1", "label"]

mapping = {'Có': 0, 'Không': 1}
df_train['label'] = df_train['label'].map(mapping)
df_val['label'] = df_val['label'].map(mapping)
df_test['label'] = df_test['label'].map(mapping)

In [ ]:
!pip install pyvi -q

In [ ]:
dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

In [ ]:
df_test

In [ ]:
dataset_train[0]

In [ ]:
from pyvi import ViTokenizer

def tokenize_function(example):
    example["sent1"] = ViTokenizer.tokenize(example["sent1"])
    example["sent2"] = ViTokenizer.tokenize(example["sent2"])
    example["ending0"] = ViTokenizer.tokenize(example["ending0"])
    example["ending1"] = ViTokenizer.tokenize(example["ending1"])
    return example

dataset_train = dataset_train.map(tokenize_function)
dataset_val = dataset_val.map(tokenize_function)
dataset_test = dataset_test.map(tokenize_function)

In [ ]:
ending_names = ["ending0", "ending1"]

def preprocess_function(examples):
    first_sentences = [[context] * 2 for context in examples["sent1"]]
    question_headers = examples["sent2"]
    second_sentences = [
        [f"{header} {examples[end][i]}" for end in ending_names] 
        for i, header in enumerate(question_headers)
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=256,   # PhoBERT-base limit
        padding="max_length"  # optional, nếu muốn batch đồng bộ
    )

    return {
        k: [v[i : i + 2] for i in range(0, len(v), 2)]
        for k, v in tokenized_examples.items()
    }

In [ ]:
tokenized_train = dataset_train.map(preprocess_function, batched=True)
tokenized_val = dataset_val.map(preprocess_function, batched=True)
tokenized_test = dataset_test.map(preprocess_function, batched=True)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Optional, Union, Dict, List

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: any
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        labels = [f["label"] for f in features] if "label" in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = [
            {k: v[i] for k, v in f.items() if k != "label"}
            for f in features
            for i in range(num_choices)
        ]

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)

        return batch

In [ ]:
collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [ ]:
!pip install evaluate -q

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

model = AutoModelForMultipleChoice.from_pretrained("./models/ViLegalBERT")

In [ ]:
training_args = TrainingArguments(
    output_dir="nli_model",
    evaluation_strategy="steps",
    eval_steps=100,
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps", 
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.predict(tokenized_test)
predictions = np.argmax(test_results.predictions, axis=1)
test_labels = test_results.label_ids

In [ ]:
test_labels

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", round(accuracy_score(test_labels, predictions)*100, 2))
print("Precision:", round(precision_score(test_labels, predictions, average='binary')*100, 2))
print("Recall:", round(recall_score(test_labels, predictions, average='binary')*100, 2))
print("F1 score:", round(f1_score(test_labels, predictions, average='binary')*100, 2))